# Training the Contrastive Backbone

This notebook demonstrates how to train the contrastive forecasting backbone from scratch.

**What this does:** We generate synthetic ARMA(p,q) time series -- autoregressive moving-average processes commonly used to model stationary time series -- and train a transformer-based model using a contrastive loss. The model learns latent representations where the predicted next-step embedding is close (high cosine similarity) to the true next-step embedding, and far from embeddings of other channels or batches.

**Key metric -- contrastive gap:** The difference between the "forecast-future" similarity (FF) and the "forecast-present" similarity (FP). A higher gap means the model has learned to distinguish future states from present states in latent space.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn.functional as F
import torch.optim as optim
from types import SimpleNamespace

from src.arma import generate_arma_batch
from src.models import ConfigurableModel, compute_metrics
from src.loss import contrastive_latent_loss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Create the model

The `ConfigurableModel` wraps a patch encoder + causal transformer + channel mixing.

- **C=4**: 4 independent channels per sample
- **H=1024**: latent embedding dimension
- **W=32**: patch/window size (each 4096-length series is split into 128 patches of 32)
- **encoder_type='gru'**: GRU encoder processes each patch as a temporal sequence
- **num_layers=12, nhead=8**: 12 transformer layers with 8 attention heads
- **ffn_mult=4**: feedforward hidden dim = 4x the embedding dim
- **depthwise_conv=3**: causal depthwise convolution with kernel size 3

In [ ]:
C, H, W = 4, 1024, 32

model = ConfigurableModel(
    C=C, H=H, W=W,
    encoder_type='gru',
    num_layers=12,
    nhead=8,
    ffn_mult=4,
    activation='gelu',
    depthwise_conv=3,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")

## Generate a fixed validation batch

We use `generate_arma_batch` to create synthetic ARMA time series. Each batch element has C=4 channels, each channel being an independent ARMA(p,q) process with p,q sampled from {1..4}.

In [ ]:
T_raw = 4096
batch_size = 8
dimension = 4  # max AR/MA order

# Fixed validation set (seed=0 for reproducibility)
x_val, _ = generate_arma_batch(batch_size=batch_size, T_raw=T_raw, C=C, seed=0, dimension=dimension)
x_val = x_val.to(device)
print(f"Validation batch shape: {x_val.shape}  # [batch, time, channels]")

## Set up training

We configure the contrastive loss and optimizer. The loss uses cosine similarity with cross-batch negatives but no cross-time negatives (since consecutive ARMA patches are nearly identical).

In [ ]:
# Loss configuration
spec = SimpleNamespace(train_configuration={
    'contrastive_divergence_temperature': 0.07,
    'contrastive_latent_noise': None,
    'loss_shape': 'cosine_similarity_batch_no_time_neg',
    'contrastive_latent_delay': 0,
})
cld = 1  # contrastive_latent_delay + 1

optimizer = optim.AdamW(model.parameters(), lr=1e-4)

## Training loop

We train for 1000 steps (a short demo -- real training uses 500k+ steps). Every 100 steps we evaluate on the fixed validation batch and print metrics.

- **FF** (forecast-future): cosine similarity between predicted and actual next-step embeddings (want high)
- **FP** (forecast-present): cosine similarity between predicted next-step and current-step embeddings (want low)
- **gap = FF - FP**: the contrastive quality metric (want high)

In [ ]:
total_steps = 1000
val_every = 100

for step in range(1, total_steps + 1):
    model.train()
    optimizer.zero_grad()

    # Generate a fresh random batch each step
    x_train, _ = generate_arma_batch(batch_size=batch_size, T_raw=T_raw, C=C, dimension=dimension)
    x_train = x_train.to(device)

    # Forward pass through the full model
    f_lat, o_lat = model(x_train)  # forecasted and original latents: [B, T, C, H]

    # Compute contrastive loss
    loss = contrastive_latent_loss((f_lat, o_lat), validation=False, spec=spec)
    loss.backward()
    optimizer.step()

    # Validate periodically
    if step % val_every == 0:
        model.eval()
        with torch.no_grad():
            fv, ov = model(x_val)
            val_ff, val_fp, val_tp, val_cb = compute_metrics(fv, ov, cld)
        gap = val_ff - val_fp
        print(f"Step {step:5d} | loss={loss.item():.4f} | "
              f"val FF={val_ff:.4f}  FP={val_fp:.4f}  gap={gap:.4f}")

## Print final metrics

In [ ]:
model.eval()
with torch.no_grad():
    fv, ov = model(x_val)
    val_ff, val_fp, val_tp, val_cb = compute_metrics(fv, ov, cld)

print(f"Final validation metrics:")
print(f"  FF (forecast-future similarity):  {val_ff:.4f}")
print(f"  FP (forecast-present similarity): {val_fp:.4f}")
print(f"  TP (true present-future sim):     {val_tp:.4f}")
print(f"  CB (cross-batch similarity):      {val_cb:.4f}")
print(f"  Gap (FF - FP):                    {val_ff - val_fp:.4f}")
print()
print("Note: with only 1000 steps the gap will be small.")
print("Full training (500k steps) reaches gap ~0.18.")

## Save checkpoint

In [ ]:
save_path = "contrastive_demo.pth"
torch.save(model.state_dict(), save_path)
print(f"Model saved to {save_path}")